# Evaluationg and storing the results in json format

In [1]:
!git clone https://github.com/UniversalDependencies/tools.git

fatal: destination path 'tools' already exists and is not an empty directory.


In [2]:
!uv pip install -r tools/requirements.txt

Checked 2 packages in 56ms


In [3]:
import subprocess
import json
import os
import glob

In [4]:
os.makedirs('metrics', exist_ok = True)

In [5]:
def extract_evaluation_metrics(gold_file, output_file, metrics_json='metrics.json'):
    """
    Runs the conllu evaluation script and extracts Precision metrics 
    for UPOS, UFeats, and UAS.
    """
    cmd = ['python', 'tools/eval.py', '-v', gold_file, output_file]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    except subprocess.CalledProcessError as e:
        print(f"Error running evaluation: {e}")
        return None

    metrics_to_extract = ['UPOS', 'UFeats', 'UAS']
    extracted_data = {}

    for line in result.stdout.splitlines():
        for key in metrics_to_extract:
            if line.startswith(key):
                # F1 is index 3
                value = float(line.split('|')[3].strip())
                extracted_data[key.lower()] = value

    with open(metrics_json, 'w') as f:
        json.dump(extracted_data, f, indent=4)
        
    return extracted_data

Extract the name of the gold dataset from the name of the parsed model output. E.g.
- parsed_data/af_afribooms-on-af_afribooms-ud-test.conllu -> test_data/af_afribooms-ud-test.conllu
- parsed_data/af_afribooms-on-de_gsd-ud-test.conllu -> test_data/de_gsd-ud-test.conllu

In [6]:
def get_gold_name(parsed_name):
    filename = os.path.basename(parsed_name)
    gold_name = filename.split('-on-')[1]

    gold_path = os.path.join("test_data", gold_name)
    return gold_path


    


In [7]:
def create_metrics_path(parsed_name):
    filename = os.path.basename(parsed_name).replace(".conllu","")
    metrics_path = os.path.join("metrics",filename + ".json")
    return metrics_path

In [8]:
for output_conllu in glob.glob("parsed_data/*.conllu"):
    gold_conllu = get_gold_name(output_conllu)
    metrics_json = create_metrics_path(output_conllu)
    extract_evaluation_metrics(gold_conllu, output_conllu, metrics_json)
